In [0]:
from pyspark.sql.functions import col, when, lit, trim, upper, current_timestamp

BRONZE_PATH = "/Volumes/workspace/ev-energy/ev-insight/bronze/ev_stations/"
SILVER_PATH = "/Volumes/workspace/ev-energy/ev-insight/silver/ev_stations/"

df = spark.read.format("delta").load(BRONZE_PATH)

df = df \
    .withColumn("lat",            col("lat").cast("double")) \
    .withColumn("lon",            col("lon").cast("double")) \
    .withColumn("num_connectors", col("num_connectors").cast("int")) \
    .withColumn("country",        trim(upper(col("country")))) \
    .withColumn("status",         trim(upper(col("status")))) \
    .withColumn("operator",       trim(col("operator")))

for c in ["state", "postcode", "operator", "town", "connector_types", "address"]:
    df = df.withColumn(c, when(col(c).isNull(), lit("Unknown")).otherwise(col(c)))

df = df \
    .filter(col("lat").isNotNull() & col("lon").isNotNull()) \
    .filter(col("country") == "GB") \
    .dropDuplicates() \
    .withColumn("cleaned_at", current_timestamp())

df.write.format("delta").mode("overwrite").save(SILVER_PATH)
print(f"Silver complete: {df.count():,} rows written")

In [0]:
df = spark.read.format("delta").load("/Volumes/workspace/ev-energy/ev-insight/bronze/ev_stations/")

df.select("country").distinct().show(50, truncate=False)